# Creation of Figures

## Step 1: Importing Libraries
All required libraries and classes are imported below.

In [ ]:
# Libraries
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from kaleido import get_chrome_sync
get_chrome_sync()

## Step 2: Loading Feature DataFrame
In the following cell, the DataFrame `features.parquet`, which was created in the notebook `features_vX.ipynb` and contains all features for our ML model, is imported.

#### <font color='red'> CHANGE THE START AND END TIMES OF THE ANALYSES IN THE FIRST LINES OF CODE: </font>

In [ ]:
# ========================= Adjust the parameters here inside =========================

START = ["2025-01-01 00:00"]  # Starting timestamp for the filtering.
STOP = ["2026-01-01 00:00"]   # Ending timestamp for the filtering.

# ========================= Adjust the parameters here inside =========================


# Load the features data.
features_df = pd.read_parquet("data/processed/features.parquet")

# Convert the taxi_start column to datetime if it is not already in that format.
features_df["taxi_start"] = pd.to_datetime(
    features_df["taxi_start"],
    utc=True
)

# Filter the dataset by START and STOP.
filtered_dfs = []

for i in range(len(START)):
    start_ts = pd.Timestamp(START[i], tz="UTC")
    stop_ts = pd.Timestamp(STOP[i], tz="UTC")

    period_df = features_df.loc[
        (features_df["taxi_start"] >= start_ts)
        & (features_df["taxi_start"] < stop_ts)
    ]

    filtered_dfs.append(period_df)

# Combine all selected periods.
features_df = pd.concat(
    filtered_dfs,
    ignore_index=True
)

In [ ]:
features_df.head()

## Step 3: Creating Figures
In the following cells, figures are created to support our thesis and provide an overview of the taxi-out situation at ZRH with regard to the available features/data.

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = 0                         # [min] Lower x-axis limit.
TAXI_TIME_MAX = 45                        # [min] Upper x-axis limit.
VIOLIN_WIDTH = 0.65                       # Width of each violin.
LINE_HALF_HEIGHT = 0.4                    # Half-height of the percentile lines per runway.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# --- Figure size ---
figure_width = 1500
figure_height = 1100

# --- Colours ---
violin_colour = "navy"
q10_colour = "green"
median_colour = "darkorange"
q90_colour = "red"

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi time from seconds to minutes.
df["taxi_time_min"] = df["taxi_time_s"] / 60.0

# Keep only rows with valid taxi times.
df = df[df["taxi_time_min"].notna()]
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted x-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    if runway_col not in df.columns:
        print(f"Column {runway_col} not found.")
        continue

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway
    tmp["runway_idx"] = RUNWAYS.index(runway)

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)

# Compute percentiles per runway.
percentiles = (
    plot_df
    .groupby(["runway", "runway_idx"])["taxi_time_min"]
    .quantile([0.10, 0.50, 0.90])
    .unstack()
    .reset_index()
    .rename(
        columns={
            0.10: "q10",
            0.50: "median",
            0.90: "q90",
        }
    )
)


# ========================= Figure =========================

fig = go.Figure()

# Add one horizontal violin per runway.
for runway in RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]
    runway_idx = RUNWAYS.index(runway)

    fig.add_trace(
        go.Violin(
            x=runway_data["taxi_time_min"],
            y0=runway_idx,
            orientation="h",
            width=VIOLIN_WIDTH,
            spanmode="hard",
            line=dict(
                color=violin_colour,
                width=2,
            ),
            fillcolor=violin_colour,
            opacity=0.75,
            points=False,
            meanline_visible=False,
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Taxi-out time: %{x:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add dashed percentile lines per runway.
for _, row in percentiles.iterrows():
    runway = row["runway"]
    runway_idx = row["runway_idx"]

    y0 = runway_idx - LINE_HALF_HEIGHT
    y1 = runway_idx + LINE_HALF_HEIGHT

    fig.add_trace(
        go.Scatter(
            x=[row["q10"], row["q10"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=q10_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"10th Percentile: {row['q10']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[row["median"], row["median"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=median_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"Median: {row['median']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[row["q90"], row["q90"]],
            y=[y0, y1],
            mode="lines",
            line=dict(
                color=q90_colour,
                width=4,
                dash="dash",
            ),
            showlegend=False,
            hovertemplate=(
                f"Runway: {runway}<br>"
                f"90th Percentile: {row['q90']:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add legend entries for the percentile lines.
fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=q10_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="10th Percentile",
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=median_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="Median",
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(
            color=q90_colour,
            width=3,
            dash="dash",
        ),
        showlegend=True,
        name="90th Percentile",
    )
)

fig.update_layout(
    title=dict(
        text="Taxi-Out Time Distribution by Runway",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=120, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Legend",
        x=0.987,
        y=0.02,
        xanchor="right",
        yanchor="bottom",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    xaxis=dict(
        title="Taxi-Out Time [min]",
        range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
        dtick=5,
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    yaxis=dict(
        title="Take-Off Runway",
        tickmode="array",
        tickvals=list(range(len(RUNWAYS))),
        ticktext=RUNWAYS,
        autorange="reversed",
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
    ),
)

fig.show()
fig.write_image("figures/taxi_out_time_distribution_by_runway.pdf")

In [ ]:
# ========================= Adjust the parameters here inside =========================

RUNWAYS = ["28", "32", "16", "10", "34"]  # Runway order in the plot.

TAXI_TIME_MIN = -1                        # [min] Lower y-axis limit.
TAXI_TIME_MAX = 46                        # [min] Upper y-axis limit.

TIME_TICK_STEP_HOURS = 1                  # X-axis tick spacing in hours.

MARKER_SIZE = 7                           # Scatter marker size in the plot.
LEGEND_MARKER_SIZE = 11                   # Scatter marker size in the legend.
MARKER_OPACITY = 0.65                     # Scatter marker opacity.

# --- Parameters for fonts ---
font_family = "Arial"
font_colour = "black"

title_size = 35
axis_title_size = 25
axis_tick_size = 25
legend_title_size = 30
legend_font_size = 25

# --- Figure size ---
figure_width = 1500
figure_height = 900

# --- Colours ---
runway_colours = {
    "28": "royalblue",
    "32": "darkred",
    "16": "darkorange",
    "10": "darkgreen",
    "34": "darkviolet",
}

# ========================= Adjust the parameters here inside =========================


df = features_df.copy()

# Convert taxi start time from UTC to Swiss local time.
df["taxi_start"] = pd.to_datetime(df["taxi_start"], utc=True)
df["taxi_start_lt"] = df["taxi_start"].dt.tz_convert("Europe/Zurich")

# Convert taxi time from seconds to minutes.
df["taxi_time_min"] = df["taxi_time_s"] / 60.0

# Keep only rows with valid taxi start times and taxi times.
df = df.dropna(
    subset=[
        "taxi_start_lt",
        "taxi_time_min",
    ]
)

# Keep only rows with non-negative taxi times.
df = df[df["taxi_time_min"] >= 0]

# Remove taxi-time outliers outside the plotted y-axis range.
df = df[
    (df["taxi_time_min"] >= TAXI_TIME_MIN)
    & (df["taxi_time_min"] <= TAXI_TIME_MAX)
].copy()

# Convert local time of day to minutes after midnight.
df["time_of_day_min"] = (
    df["taxi_start_lt"].dt.hour * 60
    + df["taxi_start_lt"].dt.minute
    + df["taxi_start_lt"].dt.second / 60.0
)

# Create local-time labels for hover information.
df["time_of_day_label"] = df["taxi_start_lt"].dt.strftime("%H:%M:%S")
df["taxi_start_lt_label"] = df["taxi_start_lt"].dt.strftime("%Y-%m-%d %H:%M:%S %Z")


# ========================= Prepare data =========================

runway_dfs = []

for runway in RUNWAYS:
    runway_col = f"TO_runway_{runway}"

    tmp = df[df[runway_col] == 1].copy()
    tmp["runway"] = runway

    runway_dfs.append(tmp)

plot_df = pd.concat(runway_dfs, ignore_index=True)


# ========================= Figure =========================

fig = go.Figure()

# Add one scatter trace per runway.
for runway in RUNWAYS:
    runway_data = plot_df[plot_df["runway"] == runway]

    fig.add_trace(
        go.Scatter(
            x=runway_data["time_of_day_min"],
            y=runway_data["taxi_time_min"],
            mode="markers",
            name=runway,
            showlegend=False,
            marker=dict(
                size=MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            customdata=runway_data[
                [
                    "time_of_day_label",
                    "taxi_start_lt_label",
                ]
            ].to_numpy(),
            hovertemplate=(
                f"Runway: {runway}<br>"
                "Local time: %{customdata[0]}<br>"
                "Taxi-out time: %{y:.2f} min"
                "<extra></extra>"
            ),
        )
    )

# Add separate dummy traces for the legend.
for runway in RUNWAYS:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name=runway,
            showlegend=True,
            marker=dict(
                size=LEGEND_MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=runway_colours[runway],
                line=dict(
                    width=0,
                ),
            ),
            hoverinfo="skip",
        )
    )

# X-axis ticks from 00:00 to 24:00.
tickvals = list(range(0, 24 * 60 + 1, TIME_TICK_STEP_HOURS * 60))
ticktext = [f"{h:02d}:00" for h in range(0, 25, TIME_TICK_STEP_HOURS)]

fig.update_layout(
    title=dict(
        text="Taxi-Out Time by Time of Day and Take-Off Runway",
        x=0.5,
        xanchor="center",
        y=0.97,
        yanchor="top",
    ),
    title_font=dict(
        size=title_size,
        family=font_family,
        color=font_colour,
    ),
    width=figure_width,
    height=figure_height,
    margin=dict(t=120, b=100, l=100),
    font=dict(
        family=font_family,
        color=font_colour,
    ),
    legend=dict(
        title="Take-Off Runway",
        x=0.015,
        y=0.974,
        xanchor="left",
        yanchor="top",
        font=dict(
            size=legend_font_size,
            family=font_family,
            color=font_colour,
        ),
        title_font=dict(
            size=legend_title_size,
            family=font_family,
            color=font_colour,
        ),
    ),
    xaxis=dict(
        title="Time of Day (Local Time)",
        range=[0, 24 * 60],
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        tickangle=90,
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
    yaxis=dict(
        title="Taxi-Out Time [min]",
        range=[TAXI_TIME_MIN, TAXI_TIME_MAX],
        title_font=dict(
            size=axis_title_size,
            family=font_family,
            color=font_colour,
        ),
        tickfont=dict(
            size=axis_tick_size,
            family=font_family,
            color=font_colour,
        ),
        showgrid=True,
    ),
)

fig.show()
fig.write_image("figures/taxi_out_time_by_time_of_day.pdf")